<h2> What is ground-truth data or gold-standard data?</h2>

A dataset that acts as the benchmark for all evaluations.

In [11]:
# fetching the raw documents

import requests

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

# flatenning the raw documents' nested structure. run 'documents_raw' in a new cell to check the structure. then delete before saving. it is very big.

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [10]:
# checking the flattened structure now after adding course name to each and every document by de-nesting it.

documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [18]:
# creating a unique ID for each document using python's MD5 to create fixed 32-character hexadecimal hash from the input string

import hashlib

def generate_document_id(doc):
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())  # encode converts into bytes, and passes it to md5 to produce a fixed 32-char hexadecimal hash
    hash_hex = hash_object.hexdigest() # converts raw bytes into readable and storable hex string
    document_id = hash_hex[:8] # raw bytes couldn't be sliced to make it short
    return document_id

<h3> Why MD5 instead of UUID for creating ID?</h3>

1. MD5 creates the same ID for the same input. UUID is random. if you reprocess the data, the same doc gets the same ID. 
2. MD5 regenerates the same ID without storing it in a DB. UUID has to be store since it cannot be regenerated from the document later.
3. MD5's ID is stable as long as the documents' fields stay the same (course + question + text). UUID would break this, because every run would assign new IDs.

In [19]:
for doc in documents:
    doc['id'] = generate_document_id(doc)

documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp',
 'id': 'c02e79ef'}

In [23]:
# grouping documents by their id and then comparing how many unique IDs are there versus total documents

from collections import defaultdict

hashes = defaultdict(list) 
# creates a dictionary where every new key starts with an empty list automatically. each key = doc_id, each value = list of docs with that ID

for doc in documents:
    doc_id = doc['id']
    hashes[doc_id].append(doc)

# If doc_id is new, hashes[doc_id] automatically becomes an empty list [].
# Then .append(doc) adds the document to that list.
# This makes grouping items by key very clean.

In [25]:
print("no. of unique IDs = ", len(hashes))
print("no. of docuements = ", len(documents))

no. of unique IDs =  947
no. of docuements =  948
